Aspect-Based Sentiment Analysis (ABSA) turns the system from “just sentiment classification” into a decision-support analytics tool. 

In [1]:
# Reproducibility Setup


import random
import numpy as np

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

print("Reproducibility setup complete")

Reproducibility setup complete


In [2]:
# Project root configuration


from pathlib import Path

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

print("Project root:", PROJECT_ROOT)

Project root: c:\Users\LENOVO\OneDrive\Documents\Strath\Masters\sentiment-analysis-tool


In [3]:
# Import Libraries


import pandas as pd
import re

from collections import defaultdict

print("Libraries loaded")

Libraries loaded


In [4]:
# Load FinBERT Results


file_path = "../results/finbert_finetuned_results.csv"

df = pd.read_csv(file_path)

print("Dataset loaded")

df.head()

Dataset loaded


,text,Actual,Predicted
0,name ive been trying to call the customer serv...,0,0
1,renew annual tpo and update certificate email ...,1,1
2,assist with having the attached form executed ...,1,1
3,i got a activation code after it got locked bt...,1,0
4,app is not working for logging ini dont know w...,0,0


In [5]:
# Map numeric sentiment labels to words for both columns
sentiment_mapping = {0: 'negative', 1: 'neutral', 2: 'positive'}

df['Actual_label'] = df['Actual'].map(sentiment_mapping)
df['Predicted_label'] = df['Predicted'].map(sentiment_mapping)

print("Numeric sentiments converted to words for Actual and Predicted columns")
df.head()

Numeric sentiments converted to words for Actual and Predicted columns


,text,Actual,Predicted,Actual_label,Predicted_label
0,name ive been trying to call the customer serv...,0,0,negative,negative
1,renew annual tpo and update certificate email ...,1,1,neutral,neutral
2,assist with having the attached form executed ...,1,1,neutral,neutral
3,i got a activation code after it got locked bt...,1,0,neutral,negative
4,app is not working for logging ini dont know w...,0,0,negative,negative


In [6]:
# Text Cleaning


def clean_text(text):

    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)

    return text


df["clean_text"] = df["text"].apply(clean_text)

df.head()

,text,Actual,Predicted,Actual_label,Predicted_label,clean_text
0,name ive been trying to call the customer serv...,0,0,negative,negative,name ive been trying to call the customer serv...
1,renew annual tpo and update certificate email ...,1,1,neutral,neutral,renew annual tpo and update certificate email ...
2,assist with having the attached form executed ...,1,1,neutral,neutral,assist with having the attached form executed ...
3,i got a activation code after it got locked bt...,1,0,neutral,negative,i got a activation code after it got locked bt...
4,app is not working for logging ini dont know w...,0,0,negative,negative,app is not working for logging ini dont know w...


In [7]:
# Define aspect categories

aspect_categories = {
    "mobile_banking_app": [
        "app",
        "application",
        "mobile app",
        "login",
        "crash",
        "password",
        "secret question"
        "mobile banking",
        "bug"
    ],

    "customer_service": [
        "customer service",
        "support",
        "agent",
        "service",
        "contact center"
        "helpdesk",
        "call center"
    ],

    "payments": [
        "payment",
        "transfer",
        "transaction",
         "sent money",
        "reversal",
        "deposit",
        "withdraw"
    ],

    "loans": [
        "loans",
        "repayment",
        "loan application",
        "facility",
        "credit"
    ],
    
    "insurance_services": [
        "insurance",
        "cover",
        "premium",
        "insurance claim",
        "place cover"
    ],

    "account_issues": [
        "account",
        "login",
        "number",
        "activation"
    ],

    "cards": [
        "card",
        "credit card",
        "debit card",
        "atm"
    ]
}

In [8]:
# Aspect Extraction Function


def extract_aspects(text):

    found_aspects = []

    for aspect, categories in aspect_categories.items():

        for keyword in categories:

            if keyword in text:
                found_aspects.append(aspect)
                break

    return found_aspects

# Extract aspects for each review
df["aspects"] = df["clean_text"].apply(extract_aspects)

df.head()

,text,Actual,Predicted,Actual_label,Predicted_label,clean_text,aspects
0,name ive been trying to call the customer serv...,0,0,negative,negative,name ive been trying to call the customer serv...,[customer_service]
1,renew annual tpo and update certificate email ...,1,1,neutral,neutral,renew annual tpo and update certificate email ...,[insurance_services]
2,assist with having the attached form executed ...,1,1,neutral,neutral,assist with having the attached form executed ...,[insurance_services]
3,i got a activation code after it got locked bt...,1,0,neutral,negative,i got a activation code after it got locked bt...,"[mobile_banking_app, account_issues]"
4,app is not working for logging ini dont know w...,0,0,negative,negative,app is not working for logging ini dont know w...,[mobile_banking_app]


In [9]:
# Link aspects with sentiment labels

absa_rows = []

for _, row in df.iterrows():

    aspects = row["aspects"]

    sentiment = row["Predicted_label"]

    for aspect in aspects:

        absa_rows.append({
            "text": row["text"],
            "aspect": aspect,
            "sentiment": sentiment
        })


absa_df = pd.DataFrame(absa_rows)

absa_df.head()

,text,aspect,sentiment
0,name ive been trying to call the customer serv...,customer_service,negative
1,renew annual tpo and update certificate email ...,insurance_services,neutral
2,assist with having the attached form executed ...,insurance_services,neutral
3,i got a activation code after it got locked bt...,mobile_banking_app,negative
4,i got a activation code after it got locked bt...,account_issues,negative


In [10]:
# Aspect-based Sentiment Summary

aspect_summary = pd.crosstab(
    absa_df["aspect"],
    absa_df["sentiment"]
)

aspect_summary


sentiment,negative,neutral,positive
aspect,,,
account_issues,101,60,0
cards,10,15,0
customer_service,41,18,9
insurance_services,10,73,0
loans,22,4,0
mobile_banking_app,139,35,19
payments,58,21,1


In [11]:
# Save results

absa_df.to_csv(
    "../results/absa_results.csv",
    index=False
)

aspect_summary.to_csv(
    "../results/absa_aspect_summary.csv"
)

print("ABSA results saved")

ABSA results saved
